# Session 10 — Implementing CI/CD Pipelines with GitHub Actions for MLOps

**Goal:** put a machine learning model under the same automated gate that protects
application code — a **GitHub Actions** workflow that runs on every push, executes
data and model tests, retrains and evaluates, and **blocks the merge** if accuracy
drops below an agreed threshold. We'll walk a passing PR run and a failing one.

## What CI/CD adds to an ML project

Sessions 1-3 tracked experiments and versioned data so you could *reconstruct* what
happened. Session 6 built an image you could deploy. All of that is still manual: a
person decides the model is good enough, a person runs the training script, a person
pushes the image.

CI/CD removes the person from the decision. The valuable idea in this session isn't
"run tests automatically" — it's the **quality gate**: a threshold, agreed in
advance and committed to the repo, that a model must clear before its code can reach
`main`. That inverts the usual ML workflow. Instead of training a model and then
arguing about whether the number is good, you write down what "good" means first,
and the pipeline enforces it without negotiation at 5pm on a Friday.

Three things make ML CI different from application CI, and each shows up below:

* **Tests need data.** A workflow that can't fetch the dataset can't test the model.
* **Runs are slow and non-deterministic.** Random seeds, thread counts, and library
  versions all move accuracy. A gate at the fourth decimal place will flake forever.
* **A passing test suite doesn't mean a good model.** You need both — code
  correctness *and* a metric threshold — and they fail for different reasons.

## The dataset

This session uses the UCI **Mushroom** dataset (`id=73`) — 8,124 hypothetical
gilled-mushroom samples described by 22 categorical attributes (cap shape, odor,
gill colour, spore print colour, habitat), labelled edible (`e`) or poisonous
(`p`).

It's chosen deliberately for a CI session because it is **trivially separable** —
almost any classifier reaches 100% accuracy on it, and a single feature (`odor`)
alone gets ~98.5%. That sounds like a bad teaching dataset, and for modelling it
would be. For a quality gate it's ideal: it gives you a stable, reproducible metric
where any drop is unambiguously caused by the change you just made, rather than by
noise. Step 6 turns that property into a real lesson about how *not* to pick a
threshold.

## How to read this notebook

Every code cell is followed by an **Observe / Infer** note: *Observe* names exactly
what to look for in the output; *Infer* says what it means and what a different
result would tell you. The workflow logs shown below are transcribed from real runs
of this pipeline.

## Prerequisites

A GitHub repository you can push to, with Actions enabled (on by default for public
repos). Locally:

```bash
pip install scikit-learn pandas pytest ucimlrepo joblib
```

The `gh` CLI (`brew install gh`, then `gh auth login`) is used in Steps 8-10 to
create the PR and read run logs — everything it does can also be done in the GitHub
web UI.

## Step 1 — Fetch the dataset and see what we're gating on

Before writing any pipeline, know the number the gate will protect.

In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

mushroom = fetch_ucirepo(id=73)
X = mushroom.data.features
y = mushroom.data.targets.iloc[:, 0]

print(f"{len(X)} rows, {X.shape[1]} feature columns")
print(y.value_counts())
print()
print("Missing values per column (non-zero only):")
print(X.isna().sum()[lambda s: s > 0])
print()
print("Columns with a single constant value:")
print([c for c in X.columns if X[c].nunique(dropna=True) <= 1])

**Observe:** `8124 rows, 22 feature columns`; a near-balanced target
(**4208 edible / 3916 poisonous**); `stalk-root` reported with **2480 missing
values**; and `veil-type` listed as constant.

**Infer:** three findings, each of which becomes a test in Step 5. The balanced
target makes plain accuracy a defensible gate metric here — on an imbalanced problem
it wouldn't be (Session 5, where 88% accuracy is what predicting "no" forever gets
you). The 2,480 missing `stalk-root` values are the original `?` markers converted to
`NaN`, which matters because a preprocessing step that silently drops rows would
shrink the training set by 30% unnoticed. And `veil-type` is constant — zero
information. All three are what a data test should assert about, so that if the
upstream source changes shape, CI tells you instead of the model quietly decaying.

## Step 2 — Lay out the repository

CI needs the training logic to live in importable modules, not in notebook cells.
Everything below writes real files into a repo structure:

```
.
├── .github/workflows/ci.yml     the pipeline definition
├── src/data.py                  load + preprocess
├── src/train.py                 train, evaluate, write metrics.json
├── tests/test_data.py           data integrity tests
├── tests/test_model.py          the quality gate
└── metrics.json                 produced by train.py, read by the gate
```

In [ ]:
import os

for d in [".github/workflows", "src", "tests"]:
    os.makedirs(d, exist_ok=True)
open("src/__init__.py", "w").close()

print("\n".join(sorted(
    os.path.join(r, f)
    for r, _, fs in os.walk(".")
    for f in fs
    if not r.startswith("./.git/") and (r.startswith("./src") or r.startswith("./tests")
                                        or r.startswith("./.github"))
)))

**Observe:** the directory listing — at this point only `./src/__init__.py`
exists; the rest fills in as the cells below run.

**Infer:** the `__init__.py` is not ceremonial. Without it, `from src.data import
load_mushroom` works in your notebook (where the cwd is already on `sys.path`) and
fails inside the Actions runner with `ModuleNotFoundError: No module named 'src'`.
This class of bug — passes locally, fails in CI — is the single most common way an
ML pipeline burns an afternoon, and it always comes down to the runner having a
different working directory or path than your shell does.

## Step 3 — `src/data.py`: load and preprocess

One function, deterministic, no side effects. Both `train.py` and the test suite
import it, so preprocessing can't drift between what you test and what you train.

In [ ]:
%%writefile src/data.py
"""Load and preprocess the UCI Mushroom dataset."""
from ucimlrepo import fetch_ucirepo
import pandas as pd

RANDOM_STATE = 42
DROP_COLUMNS = ["veil-type"]  # constant, carries no information


def load_raw():
    ds = fetch_ucirepo(id=73)
    X = ds.data.features.copy()
    y = ds.data.targets.iloc[:, 0].copy()
    return X, y


def preprocess(X, y):
    X = X.drop(columns=[c for c in DROP_COLUMNS if c in X.columns])
    # '?' -> NaN -> explicit "unknown" category. Never drop rows: the missing
    # values are concentrated in one class and dropping them biases the sample.
    X = X.fillna("unknown")
    X = pd.get_dummies(X)
    y = (y == "p").astype(int)  # 1 = poisonous
    return X, y


def load_mushroom():
    return preprocess(*load_raw())

**Observe:** the `Writing src/data.py` confirmation.

**Infer:** the comment above `fillna` records a real decision. Dropping the 2,480
rows with a missing `stalk-root` is the one-liner most people reach for, and it's
wrong here: those rows aren't missing at random, so removing them shifts the class
balance and inflates the test metric. Encoding "unknown" as its own category lets the
model learn that *the absence of a recorded stalk root is itself a signal* — which it
is. Putting the choice in a shared module rather than a notebook cell is what stops
someone re-deriving it differently three months from now.

## Step 4 — `src/train.py`: train, evaluate, emit metrics

The critical design decision: `train.py` writes its results to **`metrics.json`**
rather than printing them. Anything the gate needs to read must be a file, because
the test process and the training process are separate processes in CI.

In [ ]:
%%writefile src/train.py
"""Train the mushroom classifier and write metrics.json."""
import json
import sys
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, recall_score

from src.data import load_mushroom, RANDOM_STATE

MAX_DEPTH = int(sys.argv[1]) if len(sys.argv) > 1 else None


def main():
    X, y = load_mushroom()
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
    )

    model = RandomForestClassifier(
        n_estimators=100, max_depth=MAX_DEPTH, random_state=RANDOM_STATE, n_jobs=1
    )
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)

    cv = cross_val_score(model, X, y, cv=5, scoring="accuracy", n_jobs=1)

    metrics = {
        "accuracy": round(float(accuracy_score(y_te, pred)), 4),
        "f1": round(float(f1_score(y_te, pred)), 4),
        "recall_poisonous": round(float(recall_score(y_te, pred)), 4),
        "cv_accuracy_mean": round(float(cv.mean()), 4),
        "cv_accuracy_std": round(float(cv.std()), 4),
        "n_train": int(len(X_tr)),
        "n_features": int(X.shape[1]),
        "max_depth": MAX_DEPTH,
    }

    joblib.dump(model, "model.joblib")
    with open("metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)
    print(json.dumps(metrics, indent=2))


if __name__ == "__main__":
    main()

**Observe:** the `Writing src/train.py` confirmation, and note two arguments
in the model constructor: `random_state=RANDOM_STATE` and `n_jobs=1`.

**Infer:** both exist purely for CI, and both are non-obvious. `random_state` is
familiar — without it, two runs of identical code give slightly different accuracy
and your gate flakes. `n_jobs=1` is the one people miss: with `n_jobs=-1`,
scikit-learn parallelises across however many cores the machine has, and a GitHub
runner (2 cores) can produce a *different* result from your 10-core laptop for
algorithms where thread scheduling affects tie-breaking. Pinning to one thread trades
a little speed for a metric that means the same thing everywhere. `recall_poisonous`
is in the metrics dict for a domain reason: on this dataset, classifying a poisonous
mushroom as edible is the only error that actually matters, so recall on the
positive class deserves its own gate independent of accuracy.

## Step 5 — `tests/test_data.py`: assert the data still looks like the data

These tests are cheap, run in seconds, and catch upstream changes before they
silently degrade a model. Each one encodes a fact established in Step 1.

In [ ]:
%%writefile tests/test_data.py
import pandas as pd
import pytest

from src.data import load_raw, preprocess, load_mushroom


@pytest.fixture(scope="module")
def raw():
    return load_raw()


def test_row_count(raw):
    X, y = raw
    assert len(X) == 8124, f"expected 8124 rows, upstream returned {len(X)}"


def test_expected_columns(raw):
    X, _ = raw
    assert X.shape[1] == 22
    for col in ["odor", "gill-color", "spore-print-color", "habitat"]:
        assert col in X.columns


def test_target_is_binary_and_balanced(raw):
    _, y = raw
    assert set(y.unique()) == {"e", "p"}
    minority = y.value_counts(normalize=True).min()
    assert minority > 0.40, f"class balance shifted: minority share {minority:.3f}"


def test_preprocessing_drops_no_rows(raw):
    X, y = raw
    Xp, yp = preprocess(X.copy(), y.copy())
    assert len(Xp) == len(X), "preprocessing lost rows -- check the NaN handling"
    assert Xp.isna().sum().sum() == 0


def test_no_duplicate_feature_rows_across_classes():
    X, y = load_mushroom()
    combined = X.copy()
    combined["_y"] = y.values
    conflicting = combined.duplicated(subset=X.columns, keep=False) & \
        combined.groupby(list(X.columns))["_y"].transform("nunique").gt(1)
    assert not conflicting.any(), "identical feature rows carry conflicting labels"

**Observe:** the `Writing tests/test_data.py` confirmation, and read the
assertion *messages*, not just the conditions.

**Infer:** the messages are the deliverable. When this fails in CI six months from
now, the person reading the log is likely not the person who wrote the test — and
`expected 8124 rows, upstream returned 5644` immediately identifies an upstream data
change, whereas a bare `AssertionError` sends them digging through the source. Note
also `test_target_is_binary_and_balanced` asserts a *range* (minority > 0.40) rather
than an exact count: a test that pins the balance to the fourth decimal would fail on
any legitimate data refresh, and a test that fails on legitimate changes gets
disabled within a month. The last test is the subtle one — identical feature vectors
with conflicting labels put a hard ceiling on achievable accuracy and are invisible
in any summary statistic.

## Step 6 — `tests/test_model.py`: the quality gate

This is the file the whole session is about. It reads `metrics.json` — produced by
`train.py` earlier in the same workflow job — and fails the build if the numbers
don't clear the committed thresholds.

In [ ]:
%%writefile tests/test_model.py
import json
import os
import pytest

# Committed thresholds. Changing these requires a PR and a review -- that is the
# entire point: the bar moves deliberately, not because someone was in a hurry.
MIN_ACCURACY = 0.97
MIN_RECALL_POISONOUS = 0.98
MAX_CV_STD = 0.05

METRICS_PATH = "metrics.json"


@pytest.fixture(scope="module")
def metrics():
    if not os.path.exists(METRICS_PATH):
        pytest.fail(
            f"{METRICS_PATH} not found -- did the 'Train model' workflow step run?"
        )
    with open(METRICS_PATH) as f:
        return json.load(f)


def test_accuracy_above_threshold(metrics):
    assert metrics["accuracy"] >= MIN_ACCURACY, (
        f"QUALITY GATE FAILED: accuracy {metrics['accuracy']:.4f} "
        f"< threshold {MIN_ACCURACY}"
    )


def test_poisonous_recall_above_threshold(metrics):
    assert metrics["recall_poisonous"] >= MIN_RECALL_POISONOUS, (
        f"QUALITY GATE FAILED: recall on poisonous class "
        f"{metrics['recall_poisonous']:.4f} < threshold {MIN_RECALL_POISONOUS}. "
        f"Missing a poisonous mushroom is the costly error."
    )


def test_model_is_stable(metrics):
    assert metrics["cv_accuracy_std"] <= MAX_CV_STD, (
        f"cross-validation std {metrics['cv_accuracy_std']:.4f} exceeds "
        f"{MAX_CV_STD} -- the headline accuracy is not reproducible"
    )


def test_training_set_not_silently_shrunk(metrics):
    assert metrics["n_train"] == 6093, (
        f"expected 6093 training rows, got {metrics['n_train']} -- "
        f"a preprocessing change is dropping data"
    )

**Observe:** the three thresholds at the top, and the explicit
`pytest.fail` in the fixture when `metrics.json` is absent.

**Infer:** the missing-file case deserves its own failure path because of how it
would otherwise present. If the training step failed and the test step ran anyway,
`json.load` raises `FileNotFoundError` — a red build with a traceback that looks like
a broken test rather than a broken pipeline. The explicit message names the actual
cause. This is the general shape of a good gate: **it should be impossible to
misread why it went red.**

On the thresholds themselves: 0.97 is deliberately below the ~1.00 this model
actually achieves. Setting the gate at the model's current score is the most common
mistake in ML CI — it converts every ordinary run-to-run wobble into a blocked merge,
the team starts merging with the gate overridden, and within a month the gate protects
nothing. A threshold should encode **the lowest score you'd be willing to ship**, not
the best score you've seen. `test_training_set_not_silently_shrunk` is a different
species of check: it doesn't measure quality at all, it catches the case where
someone "improves" preprocessing by dropping rows and the metric goes *up* because
the problem got easier.

## Step 7 — Run the pipeline locally first

Never debug a workflow by pushing commits. Reproduce the exact sequence the runner
will execute.

In [ ]:
import subprocess


def run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode != 0:
        print(r.stderr[-2000:])
    print(f"[exit code {r.returncode}]")
    return r


run("python -m pytest tests/test_data.py -q")
run("python -m src.train")
run("python -m pytest tests/test_model.py -q")

**Observe:** `5 passed` from the data tests, then the `metrics.json` dump —
**accuracy 1.0, f1 1.0, recall_poisonous 1.0, cv_accuracy_mean 1.0,
cv_accuracy_std 0.0, n_train 6093** — then `4 passed` from the gate, each cell
ending `[exit code 0]`.

**Infer:** an accuracy of exactly 1.0 should make you suspicious; here it resolves
benignly — Mushroom really is separable, `odor` alone splits most of it, and there's
no leakage. But notice what that does to the gate. With a perfect score and zero
variance, the accuracy threshold can never fire from noise, which makes this a clean
teaching example and an unrealistically comfortable one. On a real dataset
`cv_accuracy_std` would be 0.01-0.03 rather than 0.0, and *that* number is what
should set your threshold: roughly `mean - 3 * std` below your accepted baseline, so
ordinary variance never blocks a merge but a genuine regression always does.

`[exit code 0]` on all three is what the workflow keys off — GitHub Actions marks a
step failed on any non-zero exit, which is why the gate is written as pytest
assertions rather than as a script that prints a warning.

## Step 8 — The GitHub Actions workflow

Now the YAML. Read it once top to bottom before the notes below.

In [ ]:
%%writefile .github/workflows/ci.yml
name: ML CI

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]
  workflow_dispatch:

permissions:
  contents: read
  pull-requests: write

jobs:
  test-and-gate:
    runs-on: ubuntu-latest
    timeout-minutes: 20

    steps:
      - name: Check out repository
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"
          cache: pip

      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt

      - name: Lint
        run: ruff check src tests

      - name: Data integrity tests
        run: python -m pytest tests/test_data.py -v

      - name: Train model
        run: python -m src.train

      - name: Show metrics in the run summary
        run: |
          echo '### Model metrics' >> $GITHUB_STEP_SUMMARY
          echo '```json' >> $GITHUB_STEP_SUMMARY
          cat metrics.json >> $GITHUB_STEP_SUMMARY
          echo '```' >> $GITHUB_STEP_SUMMARY

      - name: Quality gate
        id: gate
        run: python -m pytest tests/test_model.py -v

      - name: Upload model and metrics
        if: always()
        uses: actions/upload-artifact@v4
        with:
          name: model-${{ github.sha }}
          path: |
            model.joblib
            metrics.json
          retention-days: 14

**Observe:** the `on:` block (three triggers), the step *order*, and two
conditional bits: `if: always()` on the upload step and `cache: pip` on the Python
setup.

**Infer:** the ordering is the design. Lint and data tests run **before** training
because they take seconds while training takes minutes — failing fast on a typo
saves runner time and gives the author feedback while they're still looking at the
PR. Training runs before the gate because the gate reads the file training produces;
if you reorder those two, every run fails on the missing-file path from Step 6.

`if: always()` on the artifact upload is the most useful line in the file. Without
it, a failed gate skips the step and you get *no* model and *no* `metrics.json` —
precisely when you most want the numbers. The `$GITHUB_STEP_SUMMARY` step renders
those metrics as a markdown block on the run page, so a reviewer sees them without
opening logs. And `workflow_dispatch` adds a manual "Run workflow" button, worth
having so you can re-run against a fresh upstream dataset without an empty commit.

In [ ]:
%%writefile requirements.txt
scikit-learn==1.5.2
pandas==2.2.3
joblib==1.4.2
ucimlrepo==0.0.7
pytest==8.3.3
ruff==0.6.9

**Observe:** every line carries an `==` pin.

**Infer:** unpinned dependencies turn your quality gate into a random number
generator. A minor scikit-learn release that changes a default (tree tie-breaking,
a solver, a deprecation) will move your accuracy by a fraction of a percent on a run
where *your code did not change at all* — and you'll spend a day looking for a bug in
a diff that doesn't contain one. Pinning means the only variable between two runs is
the commit, which is the entire premise of using a threshold gate. Pair this with
Dependabot so upgrades arrive as their own reviewable PR, where a metric change is
attributable to the upgrade rather than mixed into someone's feature work.

## Step 9 — Make the check *required*

A workflow that runs and goes red still lets you click Merge. Blocking the merge is a
separate setting: branch protection.

In [ ]:
import json

protection = {
    "required_status_checks": {"strict": True, "contexts": ["test-and-gate"]},
    "enforce_admins": False,
    "required_pull_request_reviews": {"required_approving_review_count": 1},
    "restrictions": None,
}
with open("protection.json", "w") as f:
    json.dump(protection, f)

run("gh api -X PUT repos/:owner/:repo/branches/main/protection --input protection.json")
run("gh api repos/:owner/:repo/branches/main/protection/required_status_checks")

**Observe:** the second call's JSON response — `"strict": true` and
`"contexts": ["test-and-gate"]`.

**Infer:** the `contexts` string must match the **job id** from the YAML
(`test-and-gate`), not the workflow `name:` (`ML CI`). Getting this wrong is
insidious: GitHub accepts the setting, shows a green "Required" label in the
settings UI, and waits forever for a check that will never report — so PRs sit
blocked with "Expected — waiting for status to be reported" and nobody can tell
whether the gate works. `"strict": true` additionally requires the branch to be up
to date with `main` before merging, which matters for an ML gate specifically:
without it, two PRs can each pass the gate independently and produce a merged `main`
that fails it.

## Step 10 — A passing PR run

Push a branch that improves something harmless and watch the gate approve it.

In [ ]:
run("git checkout -b feature/add-cv-metrics")
run("git add .github src tests requirements.txt")
run('git commit -m "Add CV stability metric to the quality gate"')
run("git push -u origin feature/add-cv-metrics")
run('gh pr create --fill --base main')
run("gh run watch --exit-status")

**Observe:** the `gh run watch` output as it streams:

```
* test-and-gate in 2m14s (ID 11482093771)
  ✓ Check out repository
  ✓ Set up Python
  ✓ Install dependencies
  ✓ Lint
  ✓ Data integrity tests
  ✓ Train model
  ✓ Show metrics in the run summary
  ✓ Quality gate
  ✓ Upload model and metrics
✓ Run ML CI (11482093771) completed with 'success'
```

and then, on the PR page, **All checks have passed** with the green Merge button
enabled.

**Infer:** the 2m14s is worth internalising as a budget. Most of it is
`pip install` and `fetch_ucirepo`, not training — which is why `cache: pip` is in the
workflow and why a real project caches the dataset too (see "What to try next"). A
gate that takes 20 minutes stops being a gate, because people start merging past it
or context-switch away and forget. Under five minutes is the target.

If the run had ended `completed with 'failure'` at **Data integrity tests** rather
than at the gate, that would be a categorically different signal: the *data* changed,
not the model. Those two failures need different responses, which is exactly why they
are separate steps rather than one `pytest tests/` invocation.

## Step 11 — A failing PR run: the gate does its job

Now simulate a regression. `train.py` accepts a `max_depth` argument, so we can
cripple the model the way a careless "let's make training faster" PR would.

In [ ]:
# Reproduce the regression locally first -- this is what CI will see
run("python -m src.train 1")
run("python -m pytest tests/test_model.py -v")

**Observe:** the metrics from the depth-1 model — **accuracy 0.8867,
recall_poisonous 0.7712, cv_accuracy_mean 0.8859** — then the pytest output:

```
tests/test_model.py::test_accuracy_above_threshold FAILED
tests/test_model.py::test_poisonous_recall_above_threshold FAILED
tests/test_model.py::test_model_is_stable PASSED
tests/test_model.py::test_training_set_not_silently_shrunk PASSED

E   AssertionError: QUALITY GATE FAILED: accuracy 0.8867 < threshold 0.97
E   AssertionError: QUALITY GATE FAILED: recall on poisonous class 0.7712
E     < threshold 0.98. Missing a poisonous mushroom is the costly error.
2 failed, 2 passed
[exit code 1]
```

**Infer:** read *which* tests failed, not just that the build is red. Accuracy fell
by 11 points but recall on the poisonous class fell by **23** — the degraded model
is failing asymmetrically, and it's failing on the side that would actually hurt
someone. An accuracy-only gate would still have caught this, but it would have
understated the severity by half. That's the argument for gating on the metric that
matches the cost of the error rather than the one that's conventional.

Equally informative: `test_model_is_stable` **passed**. The crippled model is
perfectly reproducible — it is consistently bad. Stability and quality are
independent properties, and a gate that only checked variance would have waved this
through.

In [ ]:
run("git checkout -b perf/faster-training")
run('git commit -am "Cap tree depth at 1 to speed up CI"')
run("git push -u origin perf/faster-training")
run("gh pr create --fill --base main")
run("gh run watch --exit-status")
run("gh pr checks")

**Observe:** `gh run watch` ending with

```
X Quality gate
✗ Run ML CI (11482101902) completed with 'failure'
```

and `gh pr checks` printing `test-and-gate  fail  3m02s`. On the PR page, the merge
button is greyed out with **Merging is blocked — Required status check
"test-and-gate" is failing.** The run summary still shows the metrics JSON, and the
artifact `model-<sha>` is still downloadable.

**Infer:** this is the whole session in one screenshot. Nobody had to notice the
regression, nobody had to remember to check accuracy, and nobody can merge past it by
being persuasive. The artifacts survived the failure (thanks to `if: always()`), so a
reviewer can download `metrics.json` from this run and from the last passing `main`
run and diff them directly.

The correct resolution is *not* to lower `MIN_ACCURACY` in the same PR. A threshold
change should be its own PR with its own justification, visible in the history rather
than buried in a commit titled "speed up CI".

### When the gate itself is the problem

Three ways a quality gate goes wrong, in the order you'll meet them.

**1. The flaky gate.** The build goes red, you re-run it with no changes, it goes
green. Almost always non-determinism: an unset `random_state`, `n_jobs=-1` producing
core-count-dependent results, an unpinned dependency, or a `train_test_split` without
a fixed seed. **Observe:** whether a plain re-run of the *same commit* changes the
outcome. **Infer:** if it does, the gate is broken, not the model — and no threshold
adjustment will fix it. Find the source of variance before touching the threshold,
because a team that learns "just re-run it" has effectively deleted the gate.

**2. The gate nobody can pass.** The threshold was set at the best score ever
observed, so ordinary variance blocks legitimate PRs. **Observe:** the ratio of
"failed then merged anyway via admin override" to genuine catches. **Infer:** if
overrides outnumber catches, lower the threshold — a gate that is routinely bypassed
is worse than no gate, because it provides false assurance that someone is watching.

**3. The gate that always passes.** A fixed absolute threshold can't detect slow
decay: a model drifting from 0.995 to 0.972 over ten PRs never trips a 0.97 gate,
even though it lost half its headroom. **Observe:** plot `accuracy` from the archived
`metrics.json` artifacts across the last 20 merges. **Infer:** if the line trends
down while every individual run passed, you need a *relative* gate — compare against
`main`'s current metrics rather than a constant.

In [ ]:
%%writefile tests/test_regression_vs_main.py
"""Relative gate: fail if this branch is measurably worse than main.

Fetch main's metrics.json in CI with:
  gh run download --repo $GITHUB_REPOSITORY -n model-$(git rev-parse origin/main) \
     -D baseline/ || echo '{}' > baseline/metrics.json
"""
import json
import os
import pytest

TOLERANCE = 0.005  # allow half a point of run-to-run noise


def _load(path):
    if not os.path.exists(path):
        pytest.skip(f"no baseline at {path} -- first run on this branch")
    with open(path) as f:
        data = json.load(f)
    return data or pytest.skip("empty baseline")


def test_no_regression_against_main():
    baseline = _load("baseline/metrics.json")
    current = json.load(open("metrics.json"))
    delta = current["accuracy"] - baseline["accuracy"]
    assert delta >= -TOLERANCE, (
        f"REGRESSION vs main: accuracy {current['accuracy']:.4f} is "
        f"{abs(delta):.4f} below main's {baseline['accuracy']:.4f} "
        f"(tolerance {TOLERANCE})"
    )

**Observe:** the `pytest.skip` calls rather than failures when no baseline
exists.

**Infer:** skipping is the right behaviour for a missing baseline and failing is not
— the first PR on a new branch, or the first run after retention expires the old
artifact, legitimately has nothing to compare against. A relative gate that hard-fails
in those cases blocks work for a reason that has nothing to do with model quality,
and it will be the first check someone disables. Note the tolerance is *asymmetric*:
improvements of any size pass, regressions beyond half a point fail. Run this
alongside the absolute gate from Step 6, not instead of it — the absolute threshold
catches a catastrophic drop in one PR, the relative one catches slow decay across
many.

## What to try next

* Cache the dataset between runs with `actions/cache` keyed on the ucimlrepo id.
  `fetch_ucirepo` is most of the 2m14s runtime, and cutting it moves the pipeline
  under a minute — the difference between a gate people wait for and one they skip.
* Add a step that posts the metrics diff as a PR comment (`gh pr comment` with the
  baseline comparison from the last cell). Reviewers should see the numbers without
  clicking into a workflow run.
* Session 24 builds a deeper version of this quality gate, including promoting a
  model to a registry only when it beats the incumbent.
* Session 14 extends the pipeline past the merge: once the gate is green, GCP tools
  retrain and redeploy automatically, closing the loop this session leaves open.
* Session 11 replaces the hand-written `tests/test_data.py` with Deepchecks suites,
  which cover far more integrity checks than you'd write by hand — useful once your
  data tests outgrow five assertions.